___

# <font color= #EF4444> **Speech Radicalization Analysis - Report** </font>
#### <font color= #f0565e> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_12/05/2026._

___

## <font color= #f0565e> **Objective** </font>

This project's objective was to use a transformer to classify a tweet's specific type of radical speech. **Radical speech** refers to any message that entails an extreme form of (in this case) a negative emotion towards something, someone or a group of people. Not all radical speech is the same. For the purposes of this project, we identified **six (6)** different types of radical speech:

| **Category** | **Definition** | **Examples** | **Author** | **Reference** |
|---|---|---|---|---|
| **Hate Speech** | Expression that `attacks, insults or discriminates against a person or group based on inherent or socially recognized characteristics` (race, religion, gender, ethnicity, sexuality). The defining criterion is a speech that targets someone for who they are, not for their actions.<br>- Does NOT justify or legitimize harm (Dangerous Speech)<br>- Does NOT include purely insulting or demeaning but non-identity-based language (Extreme Speech)<br>- Does NOT focus on absolutist evaluation of political actors (Enraged Speech)<br>- Does NOT call for elimination or cleansing (Hate Grammar)<br>- Does NOT normalize state or systemic violence (Horror Grammar) | "GROUP are a bunch of psychopaths"<br>"GROUP are stupid"<br>"GROUP are pests"<br>"The religion of GROUP is too radical" | ONU | [1] |
| **Dangerous Speech** | Expression that `frames violence or harm against a target group as necessary, justified or defensive`. It does this by `portraying the target as an imminent threat` (through dehumanization, fear appeals, "purity" threats, or accusations of harm). The defining criterion is the justification or legitimation of harm. | "GROUP is a threat to our society"<br>"GROUP corrupts women and children"<br>"GROUP is barbaric and a danger to humanity"<br>"GROUP is poisoning our country" | Dangerous Speech Project | [2] |
| **Extreme Speech** | Expression that uses `highly hostile, demeaning, or exclusionary language towards a target group`, without calling for harm, removal or social control. It strongly violates social norms of civility, and is intended to provoke and demean. The defining criterion is the targetting of a group with polarizing, insulting or dehumanizing language. | "GROUP destroys everything they touch"<br>"GROUP is pathetic and laughable"<br>"GROUP behaves like animals"<br>"GROUP is disgusting and immoral" | Sahana Udupa & Matti Pohjonen | [3] |
| **Enraged Speech** | Speech that focuses on the qualities, competence or actions of political figures, institutions or organizations in an absolutist or highly polarized way. It portrays them as entirely good, bad or the sole solution to a problem. The defining criterion is the `absolutist framing of a leader, institution or organization`. | "POLITICAL_FIGURE will save us" <br>"INSTITUTION is useless"<br>"ORGANIZATION is the only solution to this plague"<br>"POLITICAL_FIGURE is failing society" | Víctor Hugo Ábrego | WIP - Personal <br>Communication |
| **Hate Grammar** | Speech that `calls for the restriction, control, removal or elimination of a target group, often framed as restoring social order, moral order or purity`. The defining criterion is a desire to force the group to stay "in place" or be removed from public or social space. | "GROUP should not be allowed on the streets"<br>"GROUP should be eliminated"<br>"GROUP is out of control"<br>"GROUP must be cleansed by God" | Gabriel Giorgi | [4] |
| **Horror Grammar** | Speech that normalizes, frames as routinary or justifies systemic violence, brutality or death, particularly state or institutional violence. It aims to desensitize audiences to violence and make cruelty seem ordinary or acceptable. The defining criterion is the `endorsement or trivialization of harm executed by institutions, the State or systemic forces`. | "The STATE response towards GROUP is so entertaining"<br>"STATE should take more strong measures towards GROUP"<br>"STATE is only doing their job"<br>"This is how the system works" | Rosana Reguillo | [5] |

These are all different categories for different types of messages, but they can sometimes overlap. Our idea for this project was to be able to use a model that can understand the nuances between the categories and be able to classify a certain text accordingly, where even humans could not tell the difference.

## <font color= #f0565e> **Data** </font>

Our data consists of approximately 6200 tweets, all regarding controversial issues or topics. The tweets we obtained talk about the following issues:

- **Asian Hate**, especially in relation to the coronavirus pandemic
- **ICE**, in the context of the agency's incursion into Minneapolis in late 2025 and early 2026
- **Islamophobia**, specifically european islamophobia due to mass migration
- **LGBT prejudice**, in general
- **Palestine / Israel**, in general

The tweets were sampled via a series of searches on common hashtags for each topic. Afterwards, they were downloaded using _Zeeschuimer_ [6], a tool developed by Digital Methods Initiative that facilitates webscrapping social media. Since the data was webscrapped directly from Twitter, a lot of noise is to be expected in the form of emojis, URLs, mentions and more. Thus, URLs, mentions, line breaks, and emojis were eliminated, as well as the search hashtags. The other hashtags were kept, eliminating the # symbol. 

The tweets themselves weren't labeled in our 6 categories, and since we had a lot of tweets, manually classifying the data was not really feasible in the given timeframe. So, we decided to automate the classification with a Large Language Model (LLM). We tested various cutting edge models but ended up using GPT 5.4, by OpenAI. It was given the following prompt:

```
I'm working on a speech radicalization project, that aims to identify categories of hate speech in a tweet dataset. For this, i defined a series of categories with examples. I need your help in classifying around 6k tweet entries, in a CSV, using these categories. The annotated dataset will be used in my task. Use a hierarchy. Hate Speech should have the least priority since it's the broader category. Dangerous Speech focuses on making the other group seem threatening and justify violence, while Extreme Speech focuses on dehumanization and hostility. Also applies to public figures, leaders and commanders. Yes, the difference is that Hate Grammar polices groups and calls for control/removal. While Hate Speech focuses on attacks based on a person's or group's identity. Yes, an important distinction for Horror Grammar is WHO is commiting the harm Take a look at the categories. Understand and read them carefully. If you have any doubts, ask before working on the annotations. Use the category, revised_definition and example columns
```

After giving the model some clarification, it returned with our tweet CSVs, annotated with the radical speech labels. Something relevant to note here is that the model overwhelmingly classified tweets as "Hate Speech" (approximately 3/4ths of the data) despite the instructions given in the initial prompt to have the least priority. This was our biggest challenge when preparating the data, and we weren't able to have the model give us more balanced results (this could, of course, also be a consequence of the data itself). We also considered having the model return "None" when it wasn't fully convinced of its label and for us to manually write it in, but too many texts were being returned without a label for this to be feasible.


For ease, all tweets were merged into a single CSV file, called ```speech_classified.csv```. The tweet labels ended up as such:

![Tweet Labels](figures/tweet_labels.png)

As we can see, the classification is heavily skewed towards the Hate Speech category, which is expected since it's the broader category and the prompt forces the model to choose a category for each tweet. The categories with the least tweets are Hate and Horror Grammar, since they are way more specific.

## <font color= #f0565e> **Modeling** </font>

#### ***Embeddings - BERTweet***

Our first step in modeling was creating word embeddings for each model. For this, we decided to use ***BERTweet***, a custom version of BERT specifically designed for english-language tweets (taking into account the informal speech found on the platform), designed by ***VinAI Research*** [7] (Fun fact: they were acquired by Qualcomm in 2025).

#### ***Baseline - Starter Models***

For the first iterations of the project's modeling, we decided to try more traditional models as a benchmark. We therefore decided to use an XGBoost Classifier, since it's a robust classifier model that isn't a Neural Network, as well as a Logistic Regression, for the simplest possible approach. We used Optuna for hyperparameter selection for XGBoost, changing the model's learning rate, n_estimators, gamma, and other parameters. We optimized for the highest possible F-1 Score macro, considering we had unbalanced classes.

#### ***Transformer Model***

Additionally, we trained the same ***BERTweet*** transformer model as a classifier. This time however, we did not use the embeddings we created previously, which were in a "frozen" state, and instead dinamically created the embeddings to optimize classification. We used a learning rate of 2e-5, and trained for 10 epochs. 

Taking into account the imbalanced classes, we created a custom **weighted trainer based on a Focal Loss function**, a "_modification of cross-entropy loss designed to address class imbalance by focusing on hard-to-classify examples_" [8] [9]. It ponderates the loss function based on class imbalance and misclassifications. For correctly classified examples, the importance is low, and for misclassified examples, the importance is high. This allows us to compensate the dataset imbalance and improve classification. 

## <font color= #f0565e> **Results** </font>

#### ***XGBoost***

In the different trials for XGBoost, we got between $0.21$ and $0.28$ in F-1 Score macro. We ran a total of 31 trials before stopping when we found out we weren't really getting useful results.

#### ***LR***

For Logistic Regression, we used cross validation to get different results for the model. Our mean F-1 Score from the 5-fold cross validation was $0.2969$ 

#### ***BERTweet***

For the transformer, we checked all the main classification metrics (Accuracy, Precision, Recall and F1). We got the following results:
- Accuracy: $0.830386$
- Precision: $0.857173$	 
- Recall: $0.830386$
- F1: $0.839299$

These are very decent results, and considerably higher than the ones we obtained using regular classification models. 

![BERTweet Confusion Matrix](figures/bertweet_cm.png)


Looking at the Confusion Matrix generated by the test of our model, we can see that most of the mistakes, unsurprisingly, come from the model not being able to specifically distinguish hate speech and other types of radical speech. Besides this, the model rarely makes mistakes between the other categories, which is pleasantly surprising.

## <font color= #f0565e> **Challenges** </font>

By far, our biggest challenge for this project was the data. As stated above, time constraints forced us to require automatic labeling of the texts we were going to use for the project. GPT 5.4, the model we used, is definitely very powerful but the classification would have been a lot better had it been made by hand, by people that fully understood the definitions and differences of the types of radical speech presented here. It ended up being a lesser issue than we anticipated, since the final product was far better than we could have imagined, but it is an area of improvement.

Another area of improvement is model age. BERTweet was originally released in May 2020. In the 6 years since, the culture of Twitter has changed a lot (especially following the site's adquisition by Elon Musk in 2022). This has changed the way people interact in the platform, and more importantly, it has affected the types of radical speech that are permitted on Twitter. Some of the worst examples were probably omitted from the corpus used when creating BERTweet due to the tighter regulations around radical speech that were present at the time, which don't exist anymore. Creating a new classifier transformer model with more modern data could be an interesting study project in the future that could provide better results for tasks like this one.

## <font color= #f0565e> **Challenges** </font>

This model could be used by think tanks to understand the types of conversations that are happening on a specific platform. Of course, Twitter would be the ideal place of study but other text-based social media websites like Reddit or Tumblr could be looked at using this tool.

This can also be a tool for law enforcement. Studies have shown that a rise in radical speech in Twitter has led to a rise in actual hate crimes happening in Spain [10], so understanding where radical spech happens, and who shares it, can be a useful preventive tool to avoid extreme and awful crimes from happening in other parts of the world. 

For our part, we created a small tool that allows a user to demo the model [here](https://huggingface.co/spaces/chofas/bertweet_rad_speech). It only allows for one text at a time, but could be adapted to allow batches of text entries to be processed at once.

# <font color= #EF4444> **Bibliography & References** </font>

[1] United Nations. (n.d.). _What is hate speech?_. _United Nations_. https://www.un.org/en/hate-speech/understanding-hate-speech/what-is-hate-speech

[2] Dangerous Speech Project. (2021, April 19). _Dangerous Speech: A Practical Guide_. _Dangerous Speech Project_. https://www.dangerousspeech.org/libraries/guide

[3] Udupa, S. & Pohjonen, M. (2019). Extreme Speech and Global Digital Cultures. _International Journal of Communication_, _13_, 3049–3067. https://ijoc.org/index.php/ijoc/article/view/9102 

[4] Giorgi,  G. (2023). Dar el salto. Odio y Mutación. _452°F Revista de Teoría de la Literatura y Literatura Comparada_, 210-218. https://doi.org/10.1344/452f.2023.28.12

[5] Reguillo, R. (2012). De las violencias: caligrafía y gramática del horror. _Desacatos_. _Revista de Ciencias Sociales_, (40), 33-46.

[6] Peeters, S., Wahl, D., Roberts, I. & Parker-Kasiewicz (2026). _Zeeschuimer_. GitHub. https://github.com/digitalmethodsinitiative/zeeschuimer

[7] VinAI Research. (2020). **BERTweet**. *Hugging Face*. https://huggingface.co/docs/transformers/en/model_doc/bertweet

[8] Tsung-Yi, L., Goyal, P., Girshick, R., He, K. & Dollár, P. (2017). _Focal Loss for Dense Object Detection_. https://arxiv.org/pdf/1708.02002

[9] Tsung-Yi, L., Goyal, P., Girshick, R., He, K. & Dollár, P. (2026). _focal-loss-pytorch_. GitHub. https://github.com/itakurah/focal-loss-pytorch

[10] Amores, J., Blanco-Herrero, D., Sánchez-Holgado, P. & Frías-Vázquez, M. (2021). **Detecting ideological hatred on Twitter. Development and evaluation of a political ideology hate speech detector in tweets in Spanish.** *Scielo*. https://www.scielo.cl/scielo.php?script=sci_arttext&pid=S0719-367X2021000200098